In [1]:
# ─── 1. IMPORTS ───────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
from scipy.stats import fisher_exact
from google.colab import drive

In [2]:
# ─── 3. LEITURA E IDENTIFICAÇÃO DAS PLANILHAS ─────────────────────────────────
# ⚙️  Ajuste PASTA_DRIVE para o caminho da pasta no seu Drive onde estão os CSVs.
# Exemplo: 'Meu Drive/dados_acidentes'  ou  'Meu Drive'  se estiverem na raiz.
PASTA_DRIVE = 'Meu Drive'   # ← altere aqui se necessário

ARQUIVOS_ESPERADOS = [
    '2023_famar.csv',
    '2024_famar.csv',
    '2023_fumes.csv',
    '2024_fumes.csv',
]

def identificar_instituicao(nome_arquivo: str) -> str:
    nome = nome_arquivo.lower()
    if 'fumes' in nome:
        return 'FUMES'
    elif 'famar' in nome:
        return 'FAMAR'
    raise ValueError(f'Não foi possível identificar a instituição em: {nome_arquivo}')

planilhas = {}   # {'nome_arquivo': DataFrame}

for nome in ARQUIVOS_ESPERADOS:
    caminho = f'{nome}'
    try:
        df = pd.read_csv(caminho)
    except FileNotFoundError:
        print(f'[AVISO] Arquivo não encontrado: {caminho}')
        continue

    df['instituicao'] = identificar_instituicao(nome)
    df['arquivo_origem'] = nome

    # Converter coluna de data (formato yyyy-dd-MM hh:mm:ss)
    # Tenta converter no formato com hora
    # Tenta converter no formato com hora
    datas_convertidas = pd.to_datetime(
        df['data_do_acidente'],
        format='%Y-%m-%d %H:%M:%S',
        errors='coerce'
    )

    # Onde falhou, tenta no formato sem hora
    datas_convertidas = datas_convertidas.fillna(
        pd.to_datetime(
            df['data_do_acidente'],
            format='%Y-%m-%d',
            errors='coerce'
        )
    )

    # Imprime datas inválidas
    mask_invalidas = (
        datas_convertidas.isna()
        & df['data_do_acidente'].notna()
    )
    if mask_invalidas.any():
        print('\nDatas com erro de conversão:')

        for idx, valor in df.loc[
            mask_invalidas,
            'data_do_acidente'
        ].items():

            print(
                f'  Linha {idx + 2}: '
                f'"{valor}" '
                f'(esperado: %Y-%m-%d %H:%M:%S '
                f'ou %Y-%m-%d)'
            )

    # Salva no dataframe
    df['data_do_acidente'] = datas_convertidas

    planilhas[nome] = df
    print(f'[OK] {nome} — {len(df)} registros | instituição: {df["instituicao"].iloc[0]}')

print(f'\nTotal de arquivos carregados: {len(planilhas)}')

[OK] 2023_famar.csv — 93 registros | instituição: FAMAR
[OK] 2024_famar.csv — 83 registros | instituição: FAMAR
[OK] 2023_fumes.csv — 9 registros | instituição: FUMES
[OK] 2024_fumes.csv — 12 registros | instituição: FUMES

Total de arquivos carregados: 4


In [3]:
# ─── 4. AGREGAÇÃO DOS BIÊNIOS ─────────────────────────────────────────────────
# Biênio FAMAR (2023 + 2024)
df_famar = pd.concat(
    [planilhas[f] for f in ['2023_famar.csv', '2024_famar.csv'] if f in planilhas],
    ignore_index=True
)

# Biênio FUMES (2023 + 2024)
df_fumes = pd.concat(
    [planilhas[f] for f in ['2023_fumes.csv', '2024_fumes.csv'] if f in planilhas],
    ignore_index=True
)

print(f'Biênio FAMAR — total de registros: {len(df_famar)}')
print(f'Biênio FUMES — total de registros: {len(df_fumes)}')

Biênio FAMAR — total de registros: 176
Biênio FUMES — total de registros: 21


In [4]:
import pandas as pd

FORMATO_DATA = '%Y-%m-%d %H:%M:%S'

print('=' * 60)
print('  RELATÓRIO DE IMPORTAÇÕES POR PLANILHA')
print('=' * 60)

total_geral = 0

for nome, df in planilhas.items():
    n = len(df)
    total_geral += n

    inst = (
        df['instituicao'].iloc[0]
        if not df.empty and 'instituicao' in df.columns
        else 'N/D'
    )

    datas_nulas = df['data_do_acidente'].isna().sum()
    datas_invalidas = []

    for idx, val in df['data_do_acidente'].items():

        # Ignora nulos
        if pd.isna(val):
            continue

        linha_planilha = idx + 2  # +1 header +1 índice zero-based
        valor_original = str(val)

        # try:
        #     pd.to_datetime(
        #         valor_original,
        #         format=FORMATO_DATA,
        #         errors='raise'
        #     )

        # except Exception:

        #     motivo = (
        #         'Formato inválido. '
        #         'Esperado: yyyy-mm-dd HH:MM:SS '
        #         f'(ex.: 2023-05-31 00:00:00)'
        #     )

        #     datas_invalidas.append({
        #         'linha': linha_planilha,
        #         'valor': valor_original,
        #         'motivo': motivo
        #     })

    print(f'\nArquivo : {nome}')
    print(f'  Instituição       : {inst}')
    print(f'  Registros totais  : {n}')
    print(f'  Datas nulas       : {datas_nulas}')
    print(f'  Datas inválidas   : {len(datas_invalidas)}')

    if datas_invalidas:
        print('\n  Datas fora do formato:')

        print(
            f'  {"Linha":>6}  '
            f'{"Valor encontrado":<30}  '
            f'Motivo'
        )

        print(
            f'  {"-" * 6}  '
            f'{"-" * 30}  '
            f'{"-" * 60}'
        )

        for erro in datas_invalidas:
            print(
                f'  {erro["linha"]:>6}  '
                f'{erro["valor"]:<30}  '
                f'{erro["motivo"]}'
            )

    else:
        print('  Datas fora de formato : nenhuma')

print(f'\n{" TOTAL GERAL ":=^60}')
print(
    f'  {total_geral} registros importados '
    f'em {len(planilhas)} arquivos'
)


  RELATÓRIO DE IMPORTAÇÕES POR PLANILHA

Arquivo : 2023_famar.csv
  Instituição       : FAMAR
  Registros totais  : 93
  Datas nulas       : 0
  Datas inválidas   : 0
  Datas fora de formato : nenhuma

Arquivo : 2024_famar.csv
  Instituição       : FAMAR
  Registros totais  : 83
  Datas nulas       : 1
  Datas inválidas   : 0
  Datas fora de formato : nenhuma

Arquivo : 2023_fumes.csv
  Instituição       : FUMES
  Registros totais  : 9
  Datas nulas       : 0
  Datas inválidas   : 0
  Datas fora de formato : nenhuma

Arquivo : 2024_fumes.csv
  Instituição       : FUMES
  Registros totais  : 12
  Datas nulas       : 0
  Datas inválidas   : 0
  Datas fora de formato : nenhuma

======================= TOTAL GERAL ========================
  197 registros importados em 4 arquivos


In [5]:
# ─── 5. FUNÇÃO: TRIMESTRE A PARTIR DA DATA ────────────────────────────────────
def adicionar_trimestre(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df['trimestre'] = (
        pd.to_datetime(
            df['data_do_acidente'],
            format='%Y-%m-%d %H:%M:%S',
            errors='coerce'
        )
        .dt.quarter
        .apply(lambda x: f'T{x}' if pd.notna(x) else None)
    )

    return df


df_famar = adicionar_trimestre(df_famar)
df_fumes = adicionar_trimestre(df_fumes)
print(df_famar)
print(
    'Trimestres identificados — FAMAR:',
    sorted(df_famar['trimestre'].dropna().unique())
)

print(
    'Trimestres identificados — FUMES:',
    sorted(df_fumes['trimestre'].dropna().unique())
)

     qtd         idade estado_civil     genero                    atividade  \
0      1  56 - 60 anos   Casado (a)  Masculino  Auxiliar de Serviços Gerais   
1      2  51 - 55 anos   Casado (a)   Feminino  Auxiliar de Serviços Gerais   
2      3  36 - 40 anos   Casado (a)   Feminino        Tecnico de Enfermagem   
3      4  31 - 35 anos   Casado (a)   Feminino        Tecnico de Enfermagem   
4      5  36 - 40 anos   Casado (a)  Masculino        Tecnico de Enfermagem   
..   ...           ...          ...        ...                          ...   
171   79  46 - 50 anos  Solteiro(a)  Masculino  Auxiliar de Serviços Gerais   
172   80  46 - 50 anos   Casado (a)  Masculino        Tecnico de Enfermagem   
173   81  46 - 50 anos   Casado (a)  Masculino        Tecnico de Enfermagem   
174   82  25 - 30 anos  Solteiro(a)  Masculino        Tecnico de Enfermagem   
175   83  18 - 24 anos  Solteiro(a)   Feminino               Enfermeiro (a)   

          turno local_de_trabalho aposentadoria loc

In [6]:
# ─── 7. RELATÓRIO TRIMESTRAL DOS BIÊNIOS ──────────────────────────────────────
def relatorio_trimestral(df: pd.DataFrame, nome_inst: str):
    total = len(df)
    print(f'\n{" " + nome_inst + " — Biênio por Trimestre ":=^60}')
    print(f'  Total do biênio: {total} registros\n')

    por_trim = (
        df.groupby('trimestre', dropna=False)
          .size()
          .reset_index(name='n')
          .sort_values('trimestre')
    )

    print(f'  {"Trimestre":<15} {"N (abs)":>10} {"% do biênio":>14}')
    print(f'  {"-"*15} {"-"*10} {"-"*14}')

    for _, row in por_trim.iterrows():
        trim = str(row['trimestre']) if pd.notna(row['trimestre']) else 'Data inválida'
        n    = int(row['n'])
        pct  = (n / total * 100) if total > 0 else 0
        print(f'  {trim:<15} {n:>10} {pct:>13.1f}%')

    print(f'  {"TOTAL":<15} {total:>10} {100.0:>13.1f}%')

relatorio_trimestral(df_famar, 'FAMAR')
relatorio_trimestral(df_fumes, 'FUMES')


=============== FAMAR — Biênio por Trimestre ===============
  Total do biênio: 176 registros

  Trimestre          N (abs)    % do biênio
  --------------- ---------- --------------
  T1.0                    49          27.8%
  T2.0                    35          19.9%
  T3.0                    52          29.5%
  T4.0                    39          22.2%
  Data inválida            1           0.6%
  TOTAL                  176         100.0%

=============== FUMES — Biênio por Trimestre ===============
  Total do biênio: 21 registros

  Trimestre          N (abs)    % do biênio
  --------------- ---------- --------------
  T1                       5          23.8%
  T2                       6          28.6%
  T3                       7          33.3%
  T4                       3          14.3%
  TOTAL                   21         100.0%


In [7]:
# ─── 8. NORMALIZAÇÃO DE TRIMESTRES + CATEGORIZAÇÃO POR FAIXA ETÁRIA ──────────

import re

# ── 8.1 Diagnóstico de colunas ────────────────────────────────────────────────
print('Colunas em df_famar:', df_famar.columns.tolist())
print('Colunas em df_fumes:', df_fumes.columns.tolist())

# ⚠️ Ajuste abaixo se o nome da coluna for diferente (ex: 'Idade', 'IDADE')
COLUNA_IDADE = 'idade'

print(f'\nAmostra "{COLUNA_IDADE}" — FAMAR:', df_famar['idade'].head(5).tolist())
print(f'Amostra "{COLUNA_IDADE}" — FUMES:', df_fumes[COLUNA_IDADE].head(5).tolist())

# ── 8.2 Corrigir trimestres (FAMAR gerou 'T1.0' por dtype float) ──────────────
def normalizar_trimestre(df):
    df = df.copy()
    df['trimestre'] = (
        df['trimestre']
        .astype(str)
        .str.replace(r'\.0$', '', regex=True)
        .where(df['trimestre'].notna(), other=pd.NA)
    )
    return df

df_famar = normalizar_trimestre(df_famar)
df_fumes = normalizar_trimestre(df_fumes)

print('\nTrimestres — FAMAR:', sorted(df_famar['trimestre'].dropna().unique()))
print('Trimestres — FUMES:', sorted(df_fumes['trimestre'].dropna().unique()))

# ── 8.3 Extração da idade média a partir do texto '41 - 45 anos' ──────────────
#
# Para cada faixa descritiva (ex: '41 - 45 anos') calcula o ponto médio:
#   média = (inicio + fim) / 2  →  ex: (41 + 45) / 2 = 43.0
#
# Esse ponto médio representa a idade típica da faixa e é usado para decidir
# em qual categoria (>=44 ou >=19 e <44) ela se enquadra.
#
# Faixas cujo intervalo cruza o limiar de 44 anos são resolvidas pela
# proximidade: qual extremo (início ou fim) está mais próximo de 44?
#   • Se o ponto médio >= 44  → '>=44 anos'
#   • Se o ponto médio >= 19 e < 44 → '>=19 e <44 anos'
#   • Demais → NaN (fora do escopo)
#
# Exemplos:
#   '41 - 45 anos' → média 43.0 → '>=19 e <44 anos'
#   '43 - 47 anos' → média 45.0 → '>=44 anos'
#   '44 - 48 anos' → média 46.0 → '>=44 anos'
#   '19 - 23 anos' → média 21.0 → '>=19 e <44 anos'
#   '10 - 14 anos' → média 12.0 → NaN

_RE_FAIXA = re.compile(
    r'^\s*(\d+)\s*[-–]\s*(\d+)\s*(?:anos?)?\s*$',
    re.IGNORECASE
)

def extrair_ponto_medio(valor) -> float:
    """
    Aceita:
      • '41 - 45 anos'  → 43.0
      • '41-45'         → 43.0
      • '44'            → 44.0   (valor numérico direto)
      • np.nan / None   → np.nan
    """
    if pd.isna(valor):
        return np.nan
    texto = str(valor).strip()
    m = _RE_FAIXA.match(texto)
    if m:
        inicio, fim = float(m.group(1)), float(m.group(2))
        return (inicio + fim) / 2.0
    # Tenta interpretar como número puro
    try:
        return float(texto)
    except ValueError:
        return np.nan

def categorizar_idade(df, col_idade=COLUNA_IDADE):
    df = df.copy()
    if col_idade not in df.columns:
        raise KeyError(
            f"Coluna '{col_idade}' não encontrada. "
            f"Ajuste COLUNA_IDADE para um dos nomes: {df.columns.tolist()}"
        )

    # Ponto médio da faixa (ou valor direto se for número)
    df['_idade_media'] = df[col_idade].apply(extrair_ponto_medio)

    faixa = pd.Series(index=df.index, dtype='object')
    m = df['_idade_media']
    faixa[m >= 44]                = '>=44 anos'
    faixa[(m >= 19) & (m < 44)]   = '>=19 e <44 anos'
    # demais (< 19 ou NaN) ficam NaN

    df['faixa_etaria'] = faixa
    df.drop(columns=['_idade_media'], inplace=True)
    return df

df_famar = categorizar_idade(df_famar)
df_fumes = categorizar_idade(df_fumes)

print('\nDistribuição de faixa etária — FAMAR:')
print(df_famar['faixa_etaria'].value_counts(dropna=False))
print('\nDistribuição de faixa etária — FUMES:')
print(df_fumes['faixa_etaria'].value_counts(dropna=False))

# ── 8.4 Mapeamento detalhado: faixa original → categoria ─────────────────────
# Útil para revisar como cada faixa descritiva foi enquadrada
print('\n── Mapeamento faixa original → categoria (FAMAR) ──')
mapa_famar = (
    df_famar[[COLUNA_IDADE, 'faixa_etaria']]
    .drop_duplicates()
    .sort_values(COLUNA_IDADE)
)
print(mapa_famar.to_string(index=False))

print('\n── Mapeamento faixa original → categoria (FUMES) ──')
mapa_fumes = (
    df_fumes[[COLUNA_IDADE, 'faixa_etaria']]
    .drop_duplicates()
    .sort_values(COLUNA_IDADE)
)
print(mapa_fumes.to_string(index=False))


Colunas em df_famar: ['qtd', 'idade', 'estado_civil', 'genero', 'atividade', 'turno', 'local_de_trabalho', 'aposentadoria', 'local_do_acidente', 'data_do_acidente', 'mes', 'dia_da_semana', 'hora_do_acidente', 'horas_trabalhadas', 'afastamento', 'tempo_de_afastamento', 'houve_internacao', 'parte_do_corpo_atingida', 'lateralidade', 'tipo_de_lesao', 'agente_causador', 'cid_10', 'local_de_atendimento_medico', 'instituicao', 'arquivo_origem', 'trimestre']
Colunas em df_fumes: ['qtd', 'idade', 'estado_civil', 'genero', 'grau_de_instrucao', 'local_de_trabalho', 'atividade', 'turno', 'local_de_trabalho_2', 'aposentadoria', 'local_do_acidente', 'data_do_acidente', 'mes', 'dia_da_semana', 'hora_do_acidente', 'horas_trabalhadas', 'afastamento', 'tempo_de_afastamento', 'houve_internacao', 'parte_do_corpo_atingida', 'lateralidade', 'tipo_de_lesao', 'agente_causador', 'cid_10', 'local_de_atendimento_medico', 'instituicao', 'arquivo_origem', 'local_de_trabalho_1', 'trimestre']

Amostra "idade" — FAMA

In [8]:
# ─── 9. SEPARAÇÃO POR TRIMESTRE E FAIXA ETÁRIA ────────────────────────────────
# Gera um dicionário com sub-DataFrames filtrados por trimestre × faixa etária,
# para cada instituição.
#
# Estrutura:
#   dados_por_trimestre = {
#       'FAMAR': { 'T1': { '>=44 anos': df, '>=19 e <44 anos': df }, ... },
#       'FUMES': { ... }
#   }

FAIXAS = ['>=44 anos', '>=19 e <44 anos']

def separar_por_trimestre_e_faixa(
    df: pd.DataFrame,
    nome_inst: str
) -> dict:
    """Retorna dicionário { trimestre: { faixa: DataFrame } }."""
    trimestres = sorted(df['trimestre'].dropna().unique())
    resultado = {}

    for trim in trimestres:
        df_trim = df[df['trimestre'] == trim].copy()
        resultado[trim] = {}

        for faixa in FAIXAS:
            resultado[trim][faixa] = df_trim[
                df_trim['faixa_etaria'] == faixa
            ].copy()

        total_trim = len(df_trim)
        total_faixas = sum(len(resultado[trim][f]) for f in FAIXAS)

        print(
            f'[{nome_inst}] {trim} | '
            f'Total no trim: {total_trim} | '
            f'Com faixa etária válida: {total_faixas} | '
            f'Excluídos (fora do escopo): {total_trim - total_faixas}'
        )
        for faixa in FAIXAS:
            print(f'       {faixa}: {len(resultado[trim][faixa])} registros')

    return resultado


print('=' * 60)
dados_famar = separar_por_trimestre_e_faixa(df_famar, 'FAMAR')
print()
dados_fumes = separar_por_trimestre_e_faixa(df_fumes, 'FUMES')

[FAMAR] T1 | Total no trim: 49 | Com faixa etária válida: 49 | Excluídos (fora do escopo): 0
       >=44 anos: 21 registros
       >=19 e <44 anos: 28 registros
[FAMAR] T2 | Total no trim: 35 | Com faixa etária válida: 35 | Excluídos (fora do escopo): 0
       >=44 anos: 18 registros
       >=19 e <44 anos: 17 registros
[FAMAR] T3 | Total no trim: 52 | Com faixa etária válida: 52 | Excluídos (fora do escopo): 0
       >=44 anos: 17 registros
       >=19 e <44 anos: 35 registros
[FAMAR] T4 | Total no trim: 39 | Com faixa etária válida: 39 | Excluídos (fora do escopo): 0
       >=44 anos: 15 registros
       >=19 e <44 anos: 24 registros

[FUMES] T1 | Total no trim: 5 | Com faixa etária válida: 5 | Excluídos (fora do escopo): 0
       >=44 anos: 5 registros
       >=19 e <44 anos: 0 registros
[FUMES] T2 | Total no trim: 6 | Com faixa etária válida: 6 | Excluídos (fora do escopo): 0
       >=44 anos: 6 registros
       >=19 e <44 anos: 0 registros
[FUMES] T3 | Total no trim: 7 | Com faixa

In [9]:
# ─── 10. TESTE EXATO DE FISHER POR TRIMESTRE ──────────────────────────────────
# Compara, dentro de cada trimestre, se há associação entre:
#   • Faixa etária (>=44 anos  vs  >=19 e <44 anos)
#   • Instituição  (FAMAR  vs  FUMES)
#
# Tabela de contingência 2×2 para cada trimestre:
#
#                 FAMAR   FUMES
#   >=44 anos      a       b
#   >=19 e <44     c       d
#
# Hipóteses:
#   H0: a proporção de cada faixa etária é independente da instituição.
#   H1: há associação entre faixa etária e instituição.
#
# Nível de significância: α = 0.05

from scipy.stats import fisher_exact

ALPHA = 0.05

resultados_fisher = []   # lista de dicionários para montar o DataFrame-resumo

# Conjunto de trimestres presentes em ambas as instituições
trimestres_famar = set(dados_famar.keys())
trimestres_fumes = set(dados_fumes.keys())
trimestres_comuns = sorted(trimestres_famar | trimestres_fumes)

print('=' * 65)
print(f'{" TESTE EXATO DE FISHER — FAIXA ETÁRIA × INSTITUIÇÃO ":^65}')
print('=' * 65)

for trim in trimestres_comuns:

    # ── Contagens por célula da tabela 2×2 ──
    a = len(dados_famar.get(trim, {}).get('>=44 anos',      pd.DataFrame()))
    c = len(dados_famar.get(trim, {}).get('>=19 e <44 anos', pd.DataFrame()))
    b = len(dados_fumes.get(trim, {}).get('>=44 anos',       pd.DataFrame()))
    d = len(dados_fumes.get(trim, {}).get('>=19 e <44 anos', pd.DataFrame()))

    tabela = [[a, b],
              [c, d]]

    # ── Fisher ──
    # alternative='two-sided': detecta qualquer direção de associação
    if (a + b + c + d) == 0:
        odds_ratio, p_valor = float('nan'), float('nan')
        interpretacao = 'Sem dados'
    else:
        odds_ratio, p_valor = fisher_exact(tabela, alternative='two-sided')
        interpretacao = (
            'Significativo (rejeita H0)'
            if p_valor < ALPHA
            else 'Não significativo (mantém H0)'
        )

    resultados_fisher.append({
        'Trimestre'               : trim,
        'FAMAR >=44'              : a,
        'FAMAR >=19 e <44'        : c,
        'FUMES >=44'              : b,
        'FUMES >=19 e <44'        : d,
        'Odds Ratio'              : round(odds_ratio, 4),
        'p-valor'                 : round(p_valor, 6),
        'Resultado (α=0.05)'      : interpretacao,
    })

    # ── Impressão detalhada ──
    print(f'\n Trimestre: {trim}')
    print(f'  Tabela de contingência 2×2:')
    print(f'  {"":20} {"FAMAR":>8} {"FUMES":>8}')
    print(f'  {"-"*38}')
    print(f'  {">=44 anos":<20} {a:>8} {b:>8}')
    print(f'  {">=19 e <44 anos":<20} {c:>8} {d:>8}')
    print(f'  {"-"*38}')
    print(f'  {"Total":<20} {a+c:>8} {b+d:>8}')
    print()
    print(f'  Odds Ratio : {odds_ratio:.4f}')
    print(f'  p-valor    : {p_valor:.6f}')
    print(f'  Resultado  : {interpretacao}')

print()
print('=' * 65)

       TESTE EXATO DE FISHER — FAIXA ETÁRIA × INSTITUIÇÃO        

 Trimestre: T1
  Tabela de contingência 2×2:
                          FAMAR    FUMES
  --------------------------------------
  >=44 anos                  21        5
  >=19 e <44 anos            28        0
  --------------------------------------
  Total                      49        5

  Odds Ratio : 0.0000
  p-valor    : 0.020800
  Resultado  : Significativo (rejeita H0)

 Trimestre: T2
  Tabela de contingência 2×2:
                          FAMAR    FUMES
  --------------------------------------
  >=44 anos                  18        6
  >=19 e <44 anos            17        0
  --------------------------------------
  Total                      35        6

  Odds Ratio : 0.0000
  p-valor    : 0.032687
  Resultado  : Significativo (rejeita H0)

 Trimestre: T3
  Tabela de contingência 2×2:
                          FAMAR    FUMES
  --------------------------------------
  >=44 anos                  17        6
  >

In [10]:
# ─── 11. TABELA-RESUMO DOS RESULTADOS ─────────────────────────────────────────
df_fisher = pd.DataFrame(resultados_fisher)

print('\n RESUMO — Teste Exato de Fisher por Trimestre')
print('=' * 65)
print(df_fisher.to_string(index=False))
print()

sig = df_fisher[df_fisher['Resultado (α=0.05)'].str.startswith('Sig')]
n_sig = len(sig)
print(
    f'Trimestres com resultado significativo (p < {ALPHA}): '
    f'{n_sig} de {len(df_fisher)}'
)
if n_sig > 0:
    for _, row in sig.iterrows():
        print(
            f'  → {row["Trimestre"]} '
            f'(OR = {row["Odds Ratio"]}, p = {row["p-valor"]})'
        )


 RESUMO — Teste Exato de Fisher por Trimestre
Trimestre  FAMAR >=44  FAMAR >=19 e <44  FUMES >=44  FUMES >=19 e <44  Odds Ratio  p-valor            Resultado (α=0.05)
       T1          21                28           5                 0       0.000 0.020800    Significativo (rejeita H0)
       T2          18                17           6                 0       0.000 0.032687    Significativo (rejeita H0)
       T3          17                35           6                 1       0.081 0.011371    Significativo (rejeita H0)
       T4          15                24           3                 0       0.000 0.071080 Não significativo (mantém H0)

Trimestres com resultado significativo (p < 0.05): 3 de 4
  → T1 (OR = 0.0, p = 0.0208)
  → T2 (OR = 0.0, p = 0.032687)
  → T3 (OR = 0.081, p = 0.011371)


In [11]:
import pandas as pd
import numpy as np
from scipy.stats import fisher_exact


# ==========================================================
# CONFIGURAÇÃO
# ==========================================================

COLUNA_ESTADO_CIVIL = 'estado_civil'
COLUNA_TRIMESTRE = 'trimestre'


# ==========================================================
# AGRUPAMENTO: SOLTEIRO vs OUTROS
# ==========================================================

def categorizar_estado_civil(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df['categoria_estado_civil'] = np.where(
        df[COLUNA_ESTADO_CIVIL].fillna('') == 'Solteiro(a)',
        'Solteiro',
        'Outros'
    )

    return df


# ==========================================================
# RESUMO POR TRIMESTRE
# ==========================================================

def gerar_resumo_trimestre(
    df: pd.DataFrame,
    instituicao: str
) -> pd.DataFrame:

    contagem = (
        df.groupby(
            [COLUNA_TRIMESTRE, 'categoria_estado_civil']
        )
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )

    # garantir colunas
    for coluna in ['Solteiro', 'Outros']:
        if coluna not in contagem.columns:
            contagem[coluna] = 0

    contagem['instituicao'] = instituicao

    contagem['total_trimestre'] = (
        contagem['Solteiro']
        + contagem['Outros']
    )

    contagem['pct_solteiro'] = (
        contagem['Solteiro']
        / contagem['total_trimestre']
        * 100
    )

    contagem['pct_outros'] = (
        contagem['Outros']
        / contagem['total_trimestre']
        * 100
    )

    return contagem[
        [
            COLUNA_TRIMESTRE,
            'instituicao',
            'Solteiro',
            'Outros',
            'total_trimestre',
            'pct_solteiro',
            'pct_outros'
        ]
    ]


# ==========================================================
# TESTE EXATO DE FISHER POR TRIMESTRE
# ==========================================================

def fisher_por_trimestre(
    resumo_famar: pd.DataFrame,
    resumo_fumes: pd.DataFrame
) -> pd.DataFrame:

    resultados = []

    trimestres = sorted(
        set(resumo_famar[COLUNA_TRIMESTRE])
        | set(resumo_fumes[COLUNA_TRIMESTRE])
    )

    for trimestre in trimestres:

        famar = resumo_famar[
            resumo_famar[COLUNA_TRIMESTRE] == trimestre
        ]

        fumes = resumo_fumes[
            resumo_fumes[COLUNA_TRIMESTRE] == trimestre
        ]

        famar_solteiro = (
            famar['Solteiro'].iloc[0]
            if len(famar) > 0 else 0
        )

        famar_outros = (
            famar['Outros'].iloc[0]
            if len(famar) > 0 else 0
        )

        fumes_solteiro = (
            fumes['Solteiro'].iloc[0]
            if len(fumes) > 0 else 0
        )

        fumes_outros = (
            fumes['Outros'].iloc[0]
            if len(fumes) > 0 else 0
        )

        tabela = [
            [famar_solteiro, famar_outros],
            [fumes_solteiro, fumes_outros]
        ]

        odds_ratio, p_value = fisher_exact(tabela)

        total_famar = (
            famar_solteiro
            + famar_outros
        )

        total_fumes = (
            fumes_solteiro
            + fumes_outros
        )

        resultados.append({
            'trimestre': trimestre,

            'famar_solteiro_n': famar_solteiro,
            'famar_outros_n': famar_outros,
            'famar_total': total_famar,
            'famar_solteiro_pct':
                round(
                    famar_solteiro
                    / total_famar
                    * 100,
                    2
                )
                if total_famar > 0 else 0,
            'famar_outros_pct':
                round(
                    famar_outros
                    / total_famar
                    * 100,
                    2
                )
                if total_famar > 0 else 0,

            'fumes_solteiro_n': fumes_solteiro,
            'fumes_outros_n': fumes_outros,
            'fumes_total': total_fumes,
            'fumes_solteiro_pct':
                round(
                    fumes_solteiro
                    / total_fumes
                    * 100,
                    2
                )
                if total_fumes > 0 else 0,
            'fumes_outros_pct':
                round(
                    fumes_outros
                    / total_fumes
                    * 100,
                    2
                )
                if total_fumes > 0 else 0,

            'odds_ratio': round(odds_ratio, 6),
            'p_value': round(p_value, 6)
        })

    return pd.DataFrame(resultados)


# ==========================================================
# EXECUÇÃO
# ==========================================================

df_famar = categorizar_estado_civil(df_famar)
df_fumes = categorizar_estado_civil(df_fumes)

resumo_famar = gerar_resumo_trimestre(
    df_famar,
    'FAMAR'
)

resumo_fumes = gerar_resumo_trimestre(
    df_fumes,
    'FUMES'
)

resultado_fisher = fisher_por_trimestre(
    resumo_famar,
    resumo_fumes
)

print(resultado_fisher)

# opcional: salvar
resultado_fisher.to_csv(
    'fisher_estado_civil_trimestre.csv',
    index=False,
    encoding='utf-8-sig'
)

  trimestre  famar_solteiro_n  famar_outros_n  famar_total  \
0        T1                17              32           49   
1        T2                 7              28           35   
2        T3                21              31           52   
3        T4                17              22           39   

   famar_solteiro_pct  famar_outros_pct  fumes_solteiro_n  fumes_outros_n  \
0               34.69             65.31                 0               5   
1               20.00             80.00                 0               6   
2               40.38             59.62                 0               7   
3               43.59             56.41                 0               3   

   fumes_total  fumes_solteiro_pct  fumes_outros_pct  odds_ratio   p_value  
0            5                 0.0             100.0         inf  0.167634  
1            6                 0.0             100.0         inf  0.566810  
2            7                 0.0             100.0         inf  0.0433